# MiDRR-Classifier — Synthetic Pipeline Demo

> **SYNTHETIC DATA — NOT REAL STUDENT SESSIONS.**  
> This notebook demonstrates the complete MiDRR training pipeline using procedurally generated
> gameplay logs from `synth.py`. No real data is needed to run it. Outputs labelled
> `(synthetic)` throughout are for pipeline validation only and must not appear in the thesis
> results chapter.

**Pipeline stages covered:**
1. Generate synthetic telemetry contract v1.1 logs
2. Build the 6-feature table (`build_feature_table`)
3. Visualise per-class feature distributions
4. Group-aware train/test split (no `player_id` leakage)
5. Train Random Forest classifier
6. Evaluate: metrics, confusion matrix
7. Permutation feature importance
8. Single-session inference example

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from midrr_classifier.synth import generate_logs
from midrr_classifier.feature_engineering import (
    build_feature_table,
    compute_evacuation_time,
    compute_decision_delay,
    compute_path_efficiency,
    compute_hazard_avoidance_ratio,
    compute_interaction_frequency,
    compute_panic_proxy,
)
from midrr_classifier.data_ingestion import split_train_test
from midrr_classifier.model_definition import MiDRRClassifier
from midrr_classifier.config import MiDRRConfig
from midrr_classifier.utils.metrics import (
    compute_classification_metrics,
    print_classification_report,
    plot_confusion_matrix,
)
from midrr_classifier.data_schema import LABEL_CLASSES

sns.set_theme(style='whitegrid', palette='Set2')
print('Imports OK')

---
## 1. Generate Synthetic Logs

Each call to `generate_logs` produces telemetry contract v1.1 rows for both FIRE and EARTHQUAKE
scenarios across all three preparedness classes. The `seed` parameter makes results reproducible.

In [ ]:
logs = generate_logs(n_per_class=40, seed=42)

print(f'Total event rows : {len(logs):,}')
print(f'Unique players   : {logs["player_id"].nunique()}')
print(f'Scenarios        : {sorted(logs["scenario_type"].unique())}')
print()
print('Event type distribution:')
print(logs['event_type'].value_counts().to_string())
print()
print('Preparedness level counts (ground-truth by construction):')
print(logs['preparedness_level'].value_counts().to_string())

In [ ]:
# Preview a single player's events
sample_pid = logs['player_id'].unique()[0]
sample_session = logs[logs['player_id'] == sample_pid].copy()
print(f'Player: {sample_pid}  |  label: {sample_session["preparedness_level"].iloc[0]}')
sample_session[['timestamp', 'event_type', 'x', 'z', 'hazard_distance']].head(12)

---
## 2. Build Feature Table

`build_feature_table` groups by `(player_id, scenario_type)` and applies the six
Chapter-3 feature formulas to each group.

In [ ]:
features = build_feature_table(logs)
print(f'Feature table shape: {features.shape}  (rows = players × scenarios)')
features.head()

In [ ]:
FEATURE_COLS = [
    'evacuation_time', 'decision_delay', 'path_efficiency_ratio',
    'hazard_avoidance_ratio', 'interaction_frequency', 'panic_proxy',
]

print('Per-class feature means (synthetic):')
features.groupby('preparedness_level')[FEATURE_COLS].mean().round(3)

---
## 3. Per-Class Feature Distributions

Box plots confirm that each feature separates the three classes as expected from the
simulation profiles. HIGH students should show low `evacuation_time`, low `decision_delay`,
high `path_efficiency_ratio`, and high `hazard_avoidance_ratio`.

In [ ]:
DISPLAY_NAMES = {
    'evacuation_time':       'Evacuation Time (s)',
    'decision_delay':        'Decision Delay (s)',
    'path_efficiency_ratio': 'Path Efficiency',
    'hazard_avoidance_ratio':'Hazard Avoidance',
    'interaction_frequency': 'Interaction Freq. (per s)',
    'panic_proxy':           'Panic Proxy (° std)',
}
CLASS_PALETTE = {'HIGH': '#2ecc71', 'MODERATE': '#f39c12', 'LOW': '#e74c3c'}
ORDER = LABEL_CLASSES  # ['HIGH', 'MODERATE', 'LOW']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), FEATURE_COLS):
    sns.boxplot(
        data=features, x='preparedness_level', y=col,
        order=ORDER, palette=CLASS_PALETTE, width=0.5, ax=ax,
    )
    ax.set_title(DISPLAY_NAMES[col], fontweight='bold')
    ax.set_xlabel('')

fig.suptitle('Per-Class Feature Distributions  (SYNTHETIC DATA)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 4. Group-Aware Train / Test Split

All sessions from a given `player_id` go entirely into train OR test — never both.
This prevents behavioural signature leakage across the split boundary.

In [ ]:
cfg = MiDRRConfig(random_state=42, test_size=0.3)

train_df, test_df = split_train_test(
    features,
    test_size=cfg.test_size,
    random_state=cfg.random_state,
)

print(f'Train : {len(train_df):>3} rows  |  {train_df["player_id"].nunique()} players')
print(f'Test  : {len(test_df):>3} rows  |  {test_df["player_id"].nunique()} players')
print()

leaked = set(train_df['player_id']) & set(test_df['player_id'])
print(f'Player ID leakage : {leaked if leaked else "NONE ✓"}')
print()

split_summary = pd.DataFrame({
    'Train': train_df['preparedness_level'].value_counts(),
    'Test' : test_df['preparedness_level'].value_counts(),
}).loc[ORDER]
print('Class distribution per split:')
print(split_summary.to_string())

---
## 5. Train Random Forest

In [ ]:
X_train = train_df[cfg.feature_cols].to_numpy()
y_train = train_df[cfg.label_col].to_numpy()
X_test  = test_df[cfg.feature_cols].to_numpy()
y_test  = test_df[cfg.label_col].to_numpy()

clf = MiDRRClassifier(cfg)
clf.build_model()
clf.fit(X_train, y_train)
print('Training complete.')

---
## 6. Evaluate: Metrics & Confusion Matrix

> **Note:** 1.000 accuracy on synthetic data is expected — the class profiles are designed
> to be separable. On real data, expect lower scores with natural class overlap.

In [ ]:
y_pred = clf.predict(X_test)
metrics = compute_classification_metrics(y_test, y_pred)

print(f'Accuracy  : {metrics["accuracy"]:.3f}')
print(f'Precision : {metrics["precision"]:.3f}  (macro)')
print(f'Recall    : {metrics["recall"]:.3f}  (macro)')
print(f'F1        : {metrics["f1"]:.3f}  (macro)')
print()
print_classification_report(y_test, y_pred, labels=LABEL_CLASSES)

In [ ]:
plot_confusion_matrix(y_test, y_pred, labels=LABEL_CLASSES)

---
## 7. Permutation Feature Importance

Permutation importance is used instead of Gini importance because Gini is biased toward
high-cardinality continuous features. The thesis should report permutation importance
(and optionally SHAP) for Chapter 4.

**Expected intuition:** `decision_delay` and `hazard_avoidance_ratio` should rank highest,
since these most directly capture BFP-mandated behaviour.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    clf.model, X_test, y_test,
    n_repeats=30, random_state=cfg.random_state, scoring='accuracy',
)

imp_df = pd.DataFrame({
    'feature':    cfg.feature_cols,
    'mean':       perm.importances_mean,
    'std':        perm.importances_std,
}).sort_values('mean', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    imp_df['feature'], imp_df['mean'],
    xerr=imp_df['std'], color='steelblue', alpha=0.85, capsize=4,
)
ax.set_xlabel('Mean accuracy decrease (± std across repeats)')
ax.set_title('Permutation Feature Importance  (SYNTHETIC DATA)', fontweight='bold')
plt.tight_layout()
plt.show()

print(imp_df.sort_values('mean', ascending=False).to_string(index=False))

---
## 8. Single-Session Inference Example

Demonstrates the path a real session would take: raw event rows →
compute features → predict label + class probabilities.
This mirrors what the Phase 7 `/predict` endpoint will do.

In [ ]:
# Pick one player's raw logs
infer_pid = logs['player_id'].unique()[3]
infer_logs = logs[logs['player_id'] == infer_pid]
true_label = infer_logs['preparedness_level'].iloc[0]

# Compute the 6 features from raw events (same functions the pipeline uses)
features_row = pd.Series({
    'evacuation_time':        compute_evacuation_time(infer_logs),
    'decision_delay':         compute_decision_delay(infer_logs),
    'path_efficiency_ratio':  compute_path_efficiency(infer_logs),
    'hazard_avoidance_ratio': compute_hazard_avoidance_ratio(infer_logs),
    'interaction_frequency':  compute_interaction_frequency(infer_logs),
    'panic_proxy':            compute_panic_proxy(infer_logs),
})

print('Feature vector:')
print(features_row.round(3).to_string())
print()

X_infer = features_row[cfg.feature_cols].to_numpy().reshape(1, -1)
pred    = clf.predict(X_infer)[0]
proba   = clf.predict_proba(X_infer)[0]

print(f'True label : {true_label}')
print(f'Predicted  : {pred}')
print()
print('Class probabilities → prepScore (0–100):')
for label, p in zip(LABEL_CLASSES, proba):
    bar = '█' * int(p * 40)
    print(f'  {label:8s}  {p:.3f}  ({int(p * 100):3d}/100)  {bar}')

---
## Notes for Real Data

When real labelled sessions arrive (Phase 3), replace `generate_logs()` with:

```python
from midrr_classifier.data_ingestion import load_raw_logs
logs = load_raw_logs('data/raw/gameplay_logs_batch01_YYYYMMDD.csv')
```

Everything downstream (`build_feature_table`, `split_train_test`, model training) runs
identically. The only difference: `preparedness_level` will be joined from the expert
labeling spreadsheet rather than being set by the generator.

**Expected real-data results:**
- Accuracy likely 0.70–0.85 (natural class overlap)
- `decision_delay` and `hazard_avoidance_ratio` should rank highest in importance
- Run 5-fold stratified CV (Phase 5) before reporting final metrics